In [13]:
import wandb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


In [14]:
wandb.login(key='6fd8eabb4a2c17f0c02bc26117f5df77c33c51a0')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [20]:
sweep_configuration_isolation_forest = {
     "method": "grid",
     "metric": {"goal": "maximize", "name": "AUC"},
     'name': "sweep-isolation-forest",
     "parameters": {
         "n_estimators": {'values': [200, 300, 500]},
         "max_samples": {'values': ['auto', 0.5, 0.7]},
         "max_features": {'values': [0.5, 1.0]},
         "bootstrap": {'values': [True, False]},
         "contamination": {'values':[ 0.1, 0.01]}
     },
}

sweep_configuration_one_class_SVM = {
     "method": "grid",
     "metric": {"goal": "maximize", "name": "AUC"},
     'name': "sweep-one-class-SVM",
     "parameters": {
         "kernel": {'values': ['rbf', 'poly', 'sigmoid']},
         "gamma": {'values': [0.01, 0.1, 'scale', 'auto']},
         "nu": {'values': [0.1, 0.3, 0.5]},
     },
}

In [ ]:
def train_one_class_SVM(kernel, gamma, nu, datatrain, dataval, y_test):
    run = wandb.init(project="aml challenge 2")
    scaler = StandardScaler()
    datatrain_normalized= scaler.fit_transform(datatrain)
    dataval_normalized = scaler.transform(dataval)

    oc_svm_reduced = OneClassSVM(kernel=kernel, gamma=gamma, nu=nu)
    oc_svm_reduced.fit(datatrain_normalized)

    test_oc_svm_scores_reduced = oc_svm_reduced.decision_function(dataval_normalized)
    val_auc_score_oc_svm_reduced = roc_auc_score(y_test, test_oc_svm_scores_reduced)
    print(f"One-Class SVM AUC (Reduced Features): {val_auc_score_oc_svm_reduced}")

    metrics = {'AUC': val_auc_score_oc_svm_reduced }
    wandb.log(metrics)

In [ ]:
def train_wrapper_one_class_SVM():
    run = wandb.init(project="aml challenge 2")
    # Split the reduced features dataframe into 80% train and 20% validation
    reduced_features_df = pd.read_csv("/kaggle/input/dataset/reduced_features.csv")
    test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

    test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
    test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

    val,test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

    val_labels = val.pop('Label')
    
    val_labels = val_labels.to_numpy()
    
    return train_one_class_SVM(
        kernel = wandb.config.kernel,
        gamma = wandb.config.gamma,
        nu=wandb.config.nu,
        datatrain = reduced_features_df,
        dataval = val,
        y_test = val_labels
    )

In [ ]:
sweep_id = wandb.sweep(sweep=sweep_configuration_one_class_SVM, project="aml challenge 2")
wandb.agent(sweep_id, function=train_wrapper_one_class_SVM);

In [5]:
best_hyper_SVM = {'gamma':0.1, 'kernel':"rbf", 'nu': 0.5 } # 0.9064286443446304

In [7]:
new_reduced_features_df = pd.read_csv("/kaggle/input/train-test-dataset/new_training_set_reduced.csv")
test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

val, test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

test_labels = test.pop('Label')

test_labels = test_labels.to_numpy()

In [9]:
scaler = StandardScaler()
datatrain_normalized= scaler.fit_transform(new_reduced_features_df)
datatest_normalized = scaler.transform(test)

oc_svm_reduced = OneClassSVM(kernel=best_hyper_SVM["kernel"], gamma=best_hyper_SVM["gamma"], nu=best_hyper_SVM["nu"])
oc_svm_reduced.fit(datatrain_normalized)

test_oc_svm_scores_reduced = oc_svm_reduced.decision_function(datatest_normalized)
val_auc_score_oc_svm_reduced = roc_auc_score(test_labels, test_oc_svm_scores_reduced)
print(f"One-Class SVM AUC (Reduced Features): {val_auc_score_oc_svm_reduced}")

One-Class SVM AUC (Reduced Features): 0.7883477469276285


In [11]:
new_reduced_features_df = pd.read_csv("/kaggle/input/dataset/reduced_features.csv")
test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

val, test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

test_labels = test.pop('Label')

test_labels = test_labels.to_numpy()

In [12]:
scaler = StandardScaler()
datatrain_normalized= scaler.fit_transform(new_reduced_features_df)
datatest_normalized = scaler.transform(test)

oc_svm_reduced = OneClassSVM(kernel=best_hyper_SVM["kernel"], gamma=best_hyper_SVM["gamma"], nu=best_hyper_SVM["nu"])
oc_svm_reduced.fit(datatrain_normalized)

test_oc_svm_scores_reduced = oc_svm_reduced.decision_function(datatest_normalized)
val_auc_score_oc_svm_reduced = roc_auc_score(test_labels, test_oc_svm_scores_reduced)
print(f"One-Class SVM AUC (Reduced Features): {val_auc_score_oc_svm_reduced}")

One-Class SVM AUC (Reduced Features): 0.8779016841147018


## Isolation Forest

In [16]:
def train_one_class_forest(n_estimators, max_samples, max_features, bootstrap, contamination, datatrain, dataval, y_val):
    run = wandb.init(project="aml challenge 2")
    scaler = StandardScaler()
    datatrain_normalized= scaler.fit_transform(datatrain)
    dataval_normalized = scaler.transform(dataval)

    oc_forest_reduced =  IsolationForest(contamination=contamination, random_state=42, n_estimators=n_estimators, bootstrap=bootstrap, max_features=max_features, max_samples=max_samples  )
    oc_forest_reduced.fit(datatrain_normalized)
    
    test_oc_forest_scores_reduced = oc_forest_reduced.decision_function(dataval_normalized)
    val_auc_score_oc_forest_reduced = roc_auc_score(y_val, test_oc_forest_scores_reduced)
    print(f"Isolation Forest AUC (Reduced Features): {val_auc_score_oc_forest_reduced}")

    metrics = {'AUC': val_auc_score_oc_forest_reduced }
    wandb.log(metrics)

In [21]:
def train_wrapper_one_class_forest():
    run = wandb.init(project="aml challenge 2")
    # Split the reduced features dataframe into 80% train and 20% validation
    reduced_features_df = pd.read_csv("/kaggle/input/dataset/reduced_features.csv")
    test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

    test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
    test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

    val,test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

    val_labels = val.pop('Label')
    
    val_labels = val_labels.to_numpy()
    
    return train_one_class_forest(
        n_estimators = wandb.config.n_estimators,
        max_samples = wandb.config.max_samples,
        max_features=wandb.config.max_features,
        bootstrap = wandb.config.bootstrap,
        contamination = wandb.config.contamination,
        datatrain = reduced_features_df,
        dataval = val,
        y_val = val_labels
    )

In [22]:
sweep_id = wandb.sweep(sweep=sweep_configuration_isolation_forest, project="aml challenge 2")
wandb.agent(sweep_id, function=train_wrapper_one_class_forest);

Create sweep with ID: jf0mpxe9
Sweep URL: https://wandb.ai/miriam-lamari2-eurecom/aml%20challenge%202/sweeps/jf0mpxe9


wandb: Agent Starting Run: d24xwrqb with config:
wandb: 	bootstrap: True
wandb: 	contamination: 0.1
wandb: 	max_features: 0.5
wandb: 	max_samples: auto
wandb: 	n_estimators: 200


Isolation Forest AUC (Reduced Features): 0.9456346978358514


AUC,▁
AUC,0.94563


wandb: Agent Starting Run: 4e6tgcur with config:
wandb: 	bootstrap: True
wandb: 	contamination: 0.1
wandb: 	max_features: 0.5
wandb: 	max_samples: auto
wandb: 	n_estimators: 300


...

AUC,▁
AUC,0.9567


wandb: Agent Starting Run: f5qa1d0o with config:
wandb: 	bootstrap: False
wandb: 	contamination: 0.01
wandb: 	max_features: 1
wandb: 	max_samples: 0.7
wandb: 	n_estimators: 500


Isolation Forest AUC (Reduced Features): 0.9556196406696611


AUC,▁
AUC,0.95562


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [23]:
best_hyper_forest = {'bootstrap': True, 'contamination': 0.1, 'max_features': 1, 'max_samples': 0.5, 'n_estimators':500  } #0.9567680685994284

In [24]:
new_reduced_features_df = pd.read_csv("/kaggle/input/train-test-dataset/new_training_set_reduced.csv")
test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

val, test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

test_labels = test.pop('Label')

test_labels = test_labels.to_numpy()

In [25]:
scaler = StandardScaler()
datatrain_normalized= scaler.fit_transform(new_reduced_features_df)
datatest_normalized = scaler.transform(test)

oc_forest_reduced =  IsolationForest(contamination=best_hyper_forest['contamination'], random_state=42, n_estimators=best_hyper_forest['n_estimators'], bootstrap=best_hyper_forest['bootstrap'], max_features=best_hyper_forest['max_features'], max_samples=best_hyper_forest['max_samples'])
oc_forest_reduced.fit(datatrain_normalized)

test_oc_forest_scores_reduced = oc_forest_reduced.decision_function(datatest_normalized)
val_auc_score_oc_forest_reduced = roc_auc_score(test_labels, test_oc_forest_scores_reduced)
print(f"Isolation Forest AUC (Reduced Features): {val_auc_score_oc_forest_reduced}")

Isolation Forest AUC (Reduced Features): 0.782430587164315


In [26]:
new_reduced_features_df = pd.read_csv("/kaggle/input/dataset/reduced_features.csv")
test_reduced_features_df = pd.read_csv("/kaggle/input/dataset/test_reduced_features.csv")

test_labels = pd.read_csv("/kaggle/input/dataset/test_labels.csv")
test_reduced_features_df = test_reduced_features_df.merge(test_labels, left_index=True, right_index=True)

val, test = train_test_split(test_reduced_features_df, test_size=0.2, random_state=42)

test_labels = test.pop('Label')

test_labels = test_labels.to_numpy()

In [27]:
scaler = StandardScaler()
datatrain_normalized= scaler.fit_transform(new_reduced_features_df)
datatest_normalized = scaler.transform(test)

oc_forest_reduced =  IsolationForest(contamination=best_hyper_forest['contamination'], random_state=42, n_estimators=best_hyper_forest['n_estimators'], bootstrap=best_hyper_forest['bootstrap'], max_features=best_hyper_forest['max_features'], max_samples=best_hyper_forest['max_samples'])
oc_forest_reduced.fit(datatrain_normalized)

test_oc_forest_scores_reduced = oc_forest_reduced.decision_function(datatest_normalized)
val_auc_score_oc_forest_reduced = roc_auc_score(test_labels, test_oc_forest_scores_reduced)
print(f"Isolation Forest AUC (Reduced Features): {val_auc_score_oc_forest_reduced}")

Isolation Forest AUC (Reduced Features): 0.9512972234865725
